![display relevant image here](path/url/to/image)
- Banner/header image

# Title
- Relevant to Data and Business Context

## Overview
- BLUF (Bottom Line Up Front)
- One paragraph summary of findings and analysis
- Frame your 'story'

## Business Understanding
- Set the stage for analysis
- Why are these findings relevant/important?
- Introduce stakeholders
- Postulate about questions you want to ask/answer

## Data Understanding
- Present the source of data
- Describe the data available
- What is relevant to keep what is not
- Present any data cleaning that needs to happen
- Null values? Type mismatches? Duplicates?

In [67]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [68]:
# EDA Code Here - Create New Cells As Needed
# LOAD DATASET
df = pd.read_csv('google_play_store_dataset.csv')

print(f"Dataset shape: {df.shape} rows, {df.shape[1]} columns")
print("First 5 rows:")
df.head()

Dataset shape: (10841, 13) rows, 13 columns
First 5 rows:


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.10,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.90,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.70,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.50,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.30,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [69]:
#Information and missing values
print("\nData types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistic's summary:")
df.describe()


Data types:
App                   str
Category              str
Rating            float64
Reviews               str
Size                  str
Installs              str
Type                  str
Price                 str
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
dtype: object

Missing Values:
App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

Statistic's summary:


,Rating
count,9367.00
mean,4.19
std,0.54
min,1.00
25%,4.00
50%,4.30
75%,4.50
max,19.00


## Data Preparation



In [70]:
# Data Prep Code Here
df_clean = df.copy()

# 0. Drop the malformed row (missing a column in the CSV, causing data misalignment).
#    This row has 'Free' in the Installs column (should be a number) and '1.9' in Category.
#    It's the only row with 'Free' in Installs, so we use that to identify it.
df_clean = df_clean[df_clean['Installs'] != 'Free']

# 1. Removes Rows that have missing Ratings
df_clean = df_clean.dropna(subset=['Rating'])

# 2. Cleans Installs & convert to an int
def clean_installs(val):
    if isinstance(val, str):
        val = val.replace(',', '').replace('+', '').strip()
        if val == 'Free' or val == '':
            return np.nan
        try:
            return int(val)
        except ValueError:
            return np.nan
    try:
        return int(val)
    except (ValueError, TypeError):
        return np.nan

df_clean['Installs'] = df_clean['Installs'].apply(clean_installs)

# 3. Cleans Price and converts it to a float
def clean_price(val):
    if isinstance(val, str):
        val = val.replace('$', '').strip()
    try:
        return float(val)
    except (ValueError, TypeError):
        return np.nan

df_clean['Price'] = df_clean['Price'].apply(clean_price)

# 4. Cleans Reviews and converts to int
def clean_reviews(val):
    if isinstance(val, str):
        val = val.strip()
        # Handle values like '3.0M' -> 3,000,000
        if val.endswith('M'):
            try:
                return int(float(val[:-1]) * 1_000_000)
            except ValueError:
                return np.nan
        val = val.replace(',', '')
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return np.nan

df_clean['Reviews'] = df_clean['Reviews'].apply(clean_reviews)

# 5. Cleans Size and converts to a num (MB)
def clean_size(val):
    if isinstance(val, str):
        val = val.strip()
        if val == 'Varies with device':
            return np.nan
        lower = val.lower()
        # Extract the numeric part
        num = re.sub(r'[^0-9.]', '', val)
        if num == '':
            return np.nan
        num = float(num)
        if 'k' in lower:
            return num / 1024  # convert KB to MB
        elif 'm' in lower:
            return num  # already MB
        else:
            # No unit - assume MB
            return num
    try:
        return float(val)
    except (ValueError, TypeError):
        return np.nan

df_clean['Size_MB'] = df_clean['Size'].apply(clean_size)

# 6. Genre cleaning
df_clean['Primary_Genre'] = df_clean['Genres'].str.split(';').str[0]

# 7. Convert 'Last Updated' values to datetime
df_clean['Last Updated'] = pd.to_datetime(df_clean['Last Updated'], errors='coerce')

# 8. Dropping columns I wont use for analysis
df_clean = df_clean.drop(columns=['Size', 'Genres', 'Current Ver', 'Android Ver'], errors='ignore')

# Check cleaned data
print("Cleaned data shape:", df_clean.shape)
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())
print("\nCleaned data preview:")
df_clean.head()

Cleaned data shape: (9366, 11)

Missing values after cleaning:
App                  0
Category             0
Rating               0
Reviews              0
Installs             0
Type                 0
Price                0
Content Rating       0
Last Updated         0
Size_MB           1637
Primary_Genre        0
dtype: int64

Cleaned data preview:


,App,Category,Rating,Reviews,Installs,Type,Price,Content Rating,Last Updated,Size_MB,Primary_Genre
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.10,159,10000,Free,0.00,Everyone,2018-01-07,19.00,Art & Design
1,Coloring book moana,ART_AND_DESIGN,3.90,967,500000,Free,0.00,Everyone,2018-01-15,14.00,Art & Design
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.70,87510,5000000,Free,0.00,Everyone,2018-08-01,8.70,Art & Design
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.50,215644,50000000,Free,0.00,Teen,2018-06-08,25.00,Art & Design
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.30,967,100000,Free,0.00,Everyone,2018-06-20,2.80,Art & Design


In [71]:
# Save cleaned data file for Tableau usage
None

## Data Analysis

In [72]:
# Analysis Code Here - if needed
None

### Link to Published Dashboard

## Conclusion

Markdown here